# RAG evaluation, stage 2: answers and judging (no GPU)

For every retrieval file from stage 1 the generator (DeepSeek-V4.1-Flash-1017) answers from the model's top-5
chunks; two reference runs bound the result: **no retrieval** (the generator's own knowledge) and **gold chunks**
(retrieval perfect). The judge (GLM-5.3-Flash, a different model family) compares each answer with the reference
answer. The report gives accuracy per system with article-clustered 95% intervals and paired differences.

The API key is the Colab secret `DI_TOKEN`; calls are cached on Drive (`ragbench/llm_cache.jsonl`), so re-running
a cell costs nothing, and every command has a hard cap on live calls.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
OUT = f'{DRIVE_ROOT}/ragbench'                       # everything produced goes here (never into the repository)
DATA_PATH = f'{DRIVE_ROOT}/data/ragturk/ragturk_formal5k.json'
REPO = 'https://github.com/mericberktas/ragbench'    # set after the repository is pushed
EXTRA = ''

# clone under a name that is NOT `ragbench`: the notebook's working directory is /content, and a folder called
# /content/ragbench would shadow the installed package as an empty namespace package
SRC = '/content/ragbench_src'
if not Path(SRC).exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, SRC], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', SRC + EXTRA], check=True)
import ragbench.data  # fails here, not later, if the install is shadowed

In [ ]:
from google.colab import userdata
os.environ['DI_TOKEN'] = userdata.get('DI_TOKEN')       # never printed, never written to Drive
from ragbench.cli import main
from ragbench.llm import LLMClient
from ragbench.cli import GEN_MODEL, JUDGE_MODEL

# smoke test: 3 live calls per model
main(['smoke', '--n', '3'])

In [ ]:
# pilot first: set PILOT to a small number, check a few answers and verdicts by eye, then set it to None
PILOT = 20
Q = f'{OUT}/questions.json'
if PILOT:
    qs = json.loads(Path(Q).read_text(encoding='utf-8'))[:PILOT]
    Q = f'{OUT}/questions_pilot.json'
    Path(Q).write_text(json.dumps(qs, ensure_ascii=False), encoding='utf-8')
    print('pilot of', PILOT, 'questions; the report below needs verdict files for exactly these questions')
OUT_RUN = f'{OUT}/pilot' if PILOT else OUT       # a pilot never mixes with the full run
common = ['--data', DATA_PATH, '--out-dir', OUT_RUN, '--questions', Q, '--max-calls', '4000']

In [ ]:
# reference runs, then one run per retrieval file
main(['generate', *common, '--condition', 'none', '--label', 'no_retrieval'])
main(['generate', *common, '--condition', 'gold', '--label', 'gold_chunks'])
for p in sorted(Path(f'{OUT}/retrieval').glob('*.json')):
    main(['generate', *common, '--condition', 'retrieved', '--retrieval', str(p), '--label', p.stem])

In [ ]:
for p in sorted(Path(f'{OUT_RUN}/answers').glob('*.jsonl')):
    main(['judge', '--data', DATA_PATH, '--out-dir', OUT_RUN, '--answers', str(p), '--max-calls', '4000'])

In [ ]:
main(['report', '--data', DATA_PATH, '--out-dir', OUT_RUN, '--questions', Q, '--reference', 'ours_pair_instr8k_s10800'])

In [ ]:
# look at a few judged answers by eye before trusting the numbers
import random
from ragbench.generate import read_jsonl
ref = {q['qid']: q for q in json.loads(Path(Q).read_text(encoding='utf-8'))}
name = 'ours_pair_instr8k_s10800__retrieved'
ans = {a['qid']: a for a in read_jsonl(f'{OUT_RUN}/answers/{name}.jsonl')}
for v in random.Random(0).sample(read_jsonl(f'{OUT_RUN}/verdicts/{name}.jsonl'), 5):
    q = ref[v['qid']]
    print(q['question'], '\n  reference:', q['answer'][:150], '\n  answer:   ', ans[v['qid']]['answer'][:150],
          '\n  verdict:  ', v['verdict'], '-', v['reason'][:120], '\n')